# Unified image & video generation with Bernini-R-1.3B and OpenVINO

[Bernini-R-1.3B](https://huggingface.co/ByteDance/Bernini-R-1.3B-Diffusers) is a unified, multi-task diffusion *renderer* from ByteDance. A single model performs **text-to-image (t2i)**, **image editing (i2i)**, **text-to-video (t2v)**, **video editing (v2v)** and **reference-image-to-video (r2v / rv2v)** generation. It is fine-tuned from [Wan2.1-1.3B](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B) and re-uses the Wan components: a `WanTransformer3DModel` diffusion transformer, a `UMT5EncoderModel` text encoder, an `AutoencoderKLWan` spatio-temporal VAE and a `UniPCMultistepScheduler`.

In this notebook we convert, optimize and run Bernini-R with OpenVINO. All of Bernini's data-dependent generation logic (the per-step denoising loop, the seven guidance modes, the source-id rotary embeddings, the per-step token assembly) is kept in python; only the heavy leaf compute -- the text encoder, the transformer block stack (`BlocksCore`), and the VAE encoder/decoder -- runs as OpenVINO static graphs. See [README.md](README.md) for the full split rationale.

#### Table of contents:
- [Prerequisites](#Prerequisites)
- [Download the model](#Download-the-model)
- [Convert and optimize the model](#Convert-and-optimize-the-model)
- [Select inference device](#Select-inference-device)
- [Build the OpenVINO pipeline](#Build-the-OpenVINO-pipeline)
- [Run the tasks](#Run-the-tasks) (t2i / t2v / i2i / v2v / r2v / rv2v)
- [Interactive demo](#Interactive-demo)

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/bernini-r-image-video/bernini-r-image-video.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top](#Table-of-contents:)

Bernini's reference code pins `diffusers==0.35.2` and `transformers==4.57.3`. The cell below installs them together with the `bernini` package and OpenVINO. **If your environment already had different versions of `diffusers`/`transformers`, restart the kernel after this cell before continuing.**

> **OpenVINO nightly**: running the VAE on GPU in fp16 for multi-frame **video** needs the newer GPU plugin in OpenVINO `>= 2026.3` (it spills the large VAE activation buffer to host instead of failing). The cell installs the nightly wheel. If you only generate images, the stable release works too.

In [1]:
%pip install -q "torch>=2.4.0" "torchvision" "diffusers==0.35.2" "transformers==4.57.3" "accelerate==0.34.2" "safetensors" "einops" "ftfy" "sentencepiece" "imageio" "imageio-ffmpeg" "decord" "tqdm" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "nncf>=2.15.0" "gradio>=4.19" "modelscope"
# modelscope pulls huggingface-hub>=1.0, which breaks transformers 4.57.3; pin it back.
%pip install -q "huggingface-hub<1.0"
# OpenVINO nightly (>= 2026.3) for fp16 video VAE on GPU; falls back to stable for images.
%pip install -q --pre -U "openvino" --index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
# Pin the Bernini commit this notebook was built against; later commits add a
# heavy `veomni` (multi-GPU) dependency that is not needed for OpenVINO inference.
# (use --force-reinstall so a previously-installed HEAD build, same version 0.1.0, is replaced)
%pip install -q --no-deps --force-reinstall "https://codeload.github.com/bytedance/Bernini/tar.gz/94a52f78b6376ddeea6d6744c25565341c12d37d"

import platform

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0"

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.3 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.1 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openvino-tokenizers 2026.4.1.0 requires openvino~=2026.4.1.dev, but you have openvino 2026.5.0.dev20261005 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("bernini-r-image-video.ipynb")

## Download the model
[back to top](#Table-of-contents:)

Download the [Bernini-R-1.3B-Diffusers](https://huggingface.co/ByteDance/Bernini-R-1.3B-Diffusers) snapshot (~12 GB). The default uses the Hugging Face Hub; set `USE_MODELSCOPE = True` if the Hub is not reachable from your network.

In [3]:
MS_MODEL_ID = "bytedance-community/Bernini-R-1.3B-Diffusers"
HF_MODEL_ID = "ByteDance/Bernini-R-1.3B-Diffusers"

In [4]:
from pathlib import Path

MODEL_DIR = Path("Bernini-R-1.3B-Diffusers")
USE_MODELSCOPE = False  # set True to download from modelscope.cn instead of huggingface.co

if not (MODEL_DIR / "config.json").exists():
    if USE_MODELSCOPE:
        from modelscope import snapshot_download

        snapshot_download(MS_MODEL_ID, local_dir=MODEL_DIR.as_posix())
    else:
        from huggingface_hub import snapshot_download

        snapshot_download(HF_MODEL_ID, local_dir=MODEL_DIR.as_posix())
print("model in", MODEL_DIR.resolve())

model in /home/amd/Desktop/OpenVINO-Notebooks-on-AMD/results/bernini-r-image-video/workdir-cpu/Bernini-R-1.3B-Diffusers


## Convert and optimize the model
[back to top](#Table-of-contents:)

Choose the weight-compression format. The `UMT5` text encoder alone is ~5 GB in FP16, so weight compression noticeably reduces the footprint. **INT8** is a good default; **INT4** is smaller but may drift slightly from the reference; **FP16** is closest to the original and the largest on disk.

In [5]:
import ipywidgets as widgets

model_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT8",
    description="Weights:",
)
model_format

Dropdown(description='Weights:', index=1, options=('FP16', 'INT8', 'INT4'), value='INT8')

In [6]:
import nncf

if model_format.value == "INT4":
    compression_config = {"mode": nncf.CompressWeightsMode.INT4_ASYM, "group_size": 64, "ratio": 1.0}
elif model_format.value == "INT8":
    compression_config = {"mode": nncf.CompressWeightsMode.INT8_ASYM}
else:
    compression_config = None

OV_DIR = Path(f"bernini_ov_{model_format.value.lower()}")
print("OpenVINO IR will be written to", OV_DIR)

OpenVINO IR will be written to bernini_ov_int8


The conversion writes one graph for the text encoder, one for the transformer block stack (`BlocksCore`, dynamic token length), and one VAE encoder/decoder graph per temporal latent length. The Wan VAE decodes the temporal axis with a causal-cache python loop, so a static graph is valid for one frame count only -- we therefore pre-build the lengths we plan to use.

`vae_latent_frames` lists the latent temporal lengths to pre-convert. Bernini first rounds the request to `n = num_frames // 4 * 4 + 1`, then the VAE works on `T_latent = (n - 1) // 4 + 1` latent frames -- so the default 20-frame video (rounded to 21) needs `T_latent = 6`. Here we pre-build `1` (images / single frame) and `6` (the 20-frame video demo). Any other length is converted automatically on first use and cached to disk, so it is only ever built once.

In [7]:
from ov_bernini_helper import convert_pipeline, num_frames_to_latent

# 1 for images, num_frames_to_latent(20) == 6 for the 20-frame video demo
convert_pipeline(MODEL_DIR, OV_DIR, compression_config=compression_config, vae_latent_frames=(1, num_frames_to_latent(20)))

KeyError: 'blocks.0.attn1.to_q.weight'

## Select inference device
[back to top](#Table-of-contents:)

You can place each component on a different device. The transformer is the compute-heavy part (run several times per step over all steps), so it benefits most from the GPU and defaults to `AUTO`.

- **Text encoder** defaults to **CPU**. The UMT5 encoder must run in fp32 (it overflows in fp16, giving a black image); compiling the ~5 GB fp32 model on GPU is slow (~80 s, the long pause you would otherwise see before the denoising bar) and competes for GPU memory, while on CPU it compiles in ~2 s and only runs twice per generation.
- **VAE** can be left on `AUTO`/GPU and runs in **fp16** there (verified to match the CPU output — no black image). For multi-frame video on GPU this relies on the OpenVINO `>= 2026.3` nightly installed above.

In [ ]:
from notebook_utils import device_widget

device_transformer = device_widget(default="AUTO", description="Transformer:")
device_text_encoder = device_widget(default="CPU", description="Text encoder:")
device_vae = device_widget(default="AUTO", description="VAE:")
widgets.VBox([device_transformer, device_text_encoder, device_vae])

## Build the OpenVINO pipeline
[back to top](#Table-of-contents:)

In [ ]:
from ov_bernini_helper import load_ov_pipeline

device_map = {
    "transformer": device_transformer.value,
    "text_encoder": device_text_encoder.value,
    "vae": device_vae.value,
}
# Everything is loaded from OV_DIR (configs/tokenizer/scheduler were copied there
# during conversion); no PyTorch weights are loaded. `source_model_dir` is only
# needed if you later request a video length whose VAE graph was not pre-built.
ov_pipe = load_ov_pipeline(OV_DIR, device_map=device_map, compression_config=compression_config, source_model_dir=MODEL_DIR)

## Run the tasks
[back to top](#Table-of-contents:)

Bernini-R is multi-task. Each task uses a specific guidance mode and system prompt; these are pre-defined in `ov_bernini_helper.TASK_GUIDANCE` / `TASK_SYSTEM_PROMPT` (matching the reference `bytedance/Bernini` testcases) and exposed through this small helper. The pipeline writes the result to `output_path` and returns the path -- a PNG for a single frame, an mp4 otherwise.

| task | what it does | extra input |
|------|--------------|-------------|
| `t2i` | text -> image | - |
| `t2v` | text -> video | - |
| `i2i` | image editing | `image` |
| `v2v` | video editing | `video` |
| `r2v` | reference image(s) -> video | `images` |
| `rv2v` | reference image(s) + source video -> video | `video`, `images` |

In [ ]:
from ov_bernini_helper import TASK_GUIDANCE, TASK_SYSTEM_PROMPT


def generate(task, prompt, num_frames=1, steps=40, seed=42, output_path=None, **inputs):
    ext = "png" if (task in ("t2i", "i2i") or num_frames == 1) else "mp4"
    return ov_pipe(
        prompt=prompt,
        guidance_mode=TASK_GUIDANCE[task],
        system_prompt=TASK_SYSTEM_PROMPT[task],
        num_frames=num_frames,
        max_image_size=624,
        height=480,
        width=832,
        num_inference_steps=steps,
        seed=seed,
        output_path=output_path or f"{task}.{ext}",
        **inputs,
    )

### Text-to-image

In [ ]:
from PIL import Image

out_path = generate(
    "t2i",
    "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k",
    num_frames=1,
)
Image.open(out_path)

### Text-to-video
Uses the same transformer/VAE graphs with `num_frames > 1`. Following the Wan notebooks, the demo defaults to **20 frames** (the length whose VAE graph was pre-built above) and a low step count for video. Generating at another frame count compiles and caches the matching VAE graph on first use.

In [ ]:
from IPython.display import Video

video_path = generate("t2v", "A cat walks on the grass, realistic", num_frames=20, steps=4, seed=42)
Video(video_path, embed=True)

### Image editing (i2i)
Edit a source image with an instruction prompt. The VAE encoder embeds the source image, and the diffusion sampler renders the edited result.

In [ ]:
from notebook_utils import download_file

src = download_file(
    "https://raw.githubusercontent.com/bytedance/Bernini/main/assets/testcases/i2i/source.png",
    "i2i_source.png",
)
edited = generate(
    "i2i",
    "Change the season to a snowy winter scene, keeping the composition unchanged.",
    num_frames=1,
    image=str(src),
)
Image.open(edited)

### Video editing (v2v)
Edit a source video with an instruction prompt (e.g. add or modify an object) while preserving the rest of the scene.

In [ ]:
src_v = download_file(
    "https://raw.githubusercontent.com/bytedance/Bernini/main/assets/testcases/v2v/source_case1.mp4",
    "v2v_source.mp4",
)
edited_v = generate(
    "v2v",
    "Add a realistic snowman beside the path, matching the lighting and snow.",
    num_frames=20,
    steps=4,
    video=str(src_v),
)
Video(edited_v, embed=True)

> Reference-to-video (`r2v`) and reference+video editing (`rv2v`) work the same way -- pass `images=[...]` (and `video=...` for `rv2v`). They are available in the interactive demo below.

## Interactive demo
[back to top](#Table-of-contents:)

The Gradio demo exposes all six tasks. Image / video conditioning inputs appear for the tasks that use them.